# Coleta de dados do vlr.gg — TCC (Google Colab)

Este notebook roda o scraper do projeto `vlr_scraper` no Colab e grava **tudo numa pasta do Google Drive**:

| Pasta no Drive | Conteúdo | Formato |
|---|---|---|
| `vlr_dados/raw/` | páginas baixadas (cache da coleta, permite retomar) | `.html.gz` |
| `vlr_dados/processed/` | tabelas finais (`rounds`, `mapas`, `partidas`...) | Parquet |
| `vlr_dados/amostra/` | tabelas de exemplo, para inspeção rápida | CSV |
| `vlr_dados/logs/` | log de cada execução e `falhas.parquet` | texto / Parquet |

**Código:** por padrão, o notebook baixa o código direto do GitHub (`isa-dib/vlr_scraper`); não é preciso enviar nada ao Drive. Se preferir usar uma cópia no Drive, mude `FONTE_CODIGO` para `"drive"` na célula 2.

**Se o Colab desconectar:** rode de novo as células 1, 2 e 4 e depois a célula de coleta. O que já está em `raw/` não é baixado outra vez.

## 1. Montar o Google Drive e definir as pastas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA_TCC    = '/content/drive/MyDrive/TCC'
PASTA_DADOS  = f'{PASTA_TCC}/vlr_dados'        # onde ficam todas as saídas
PASTA_CODIGO = f'{PASTA_TCC}/vlr_scraper'      # só usada com FONTE_CODIGO = "drive" (pasta ou .zip)

## 2. Baixar o código e instalar dependências

O código vai para o disco local do Colab (`/content`), que é mais rápido. Os **dados** continuam indo para o Drive.

- `FONTE_CODIGO = "github"` (padrão): clona a versão mais recente do repositório.
- `FONTE_CODIGO = "drive"`: usa a pasta `vlr_scraper` ou um `.zip` em `MyDrive/TCC/`. Pastas aninhadas e zips baixados do GitHub (`vlr_scraper-main/`) também funcionam.

A célula termina importando o pacote `vlr`. Se algo der errado, ela diz o que faltou.

In [ ]:
import importlib, os, shutil, subprocess, sys, zipfile
from pathlib import Path

FONTE_CODIGO = "github"                       # "github" ou "drive"
REPO = "https://github.com/isa-dib/vlr_scraper"
DESTINO = Path("/content/codigo_vlr")

shutil.rmtree(DESTINO, ignore_errors=True)
if FONTE_CODIGO == "github":
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(DESTINO)], check=True)
elif os.path.isdir(PASTA_CODIGO):
    shutil.copytree(PASTA_CODIGO, DESTINO, ignore=shutil.ignore_patterns("data", "__pycache__", ".pytest_cache", ".git"))
elif os.path.exists(PASTA_CODIGO + ".zip"):
    with zipfile.ZipFile(PASTA_CODIGO + ".zip") as z:
        z.extractall(DESTINO)
else:
    raise FileNotFoundError(f"Não achei {PASTA_CODIGO} nem {PASTA_CODIGO}.zip no Drive. "
                            "Envie o projeto para lá ou use FONTE_CODIGO = 'github'.")

# localiza a raiz do projeto (a pasta que contém vlr/__init__.py), mesmo se estiver aninhada
raizes = sorted({p.parent.parent for p in DESTINO.rglob("vlr/__init__.py")}, key=lambda p: len(p.parts))
if not raizes:
    raise FileNotFoundError(f"O código copiado para {DESTINO} não tem a pasta vlr/. "
                            "Confira se a pasta enviada é a do projeto (com vlr/, tests/ e requirements.txt).")
LOCAL = str(raizes[0])
os.chdir(LOCAL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

for m in [m for m in sys.modules if m == "vlr" or m.startswith("vlr.")]:
    del sys.modules[m]                        # garante a versão recém-copiada se a célula for rodada de novo
if LOCAL not in sys.path:
    sys.path.insert(0, LOCAL)
importlib.invalidate_caches()
import vlr
print(f"código em {LOCAL} (fonte: {FONTE_CODIGO})")

## 3. Rodar os testes (opcional, ~10 s)

Confere o parser contra páginas reais salvas no projeto. Não acessa o site e não grava no Drive.

In [ ]:
!python -m pytest -q tests

## 4. Preparar a pasta de dados

`min_delay` e `max_delay` são o intervalo (em segundos) entre requisições ao vlr.gg. O padrão de 3 a 6 s é o mais seguro para coletas longas.

In [ ]:
from vlr import colab
colab.preparar(PASTA_DADOS, min_delay=3, max_delay=6)

## 5. Amostra: ver o que é extraído, sem acessar o site

Gera em `vlr_dados/amostra/` os mesmos arquivos da coleta real, **em CSV**, a partir de duas partidas reais salvas no projeto.

In [ ]:
amostra = colab.amostra()
amostra['rounds'].head()

In [ ]:
amostra['mapas']

## 6. Listar os eventos concluídos (todas as páginas do tier 60)

In [ ]:
eventos = colab.listar_eventos()
eventos[['event_id', 'event_name', 'dates', 'region', 'page']]

## 7. Sondagem de 2D replay (opcional, demorada)

Testa a 1ª e a última partida de cada evento. Para todos os eventos, leva ~1 a 2 h com o intervalo padrão. O resultado já está em `docs/sondagem_replay.csv` (out/2026); rode só se quiser atualizar.

In [ ]:
# sondagem = colab.sondar()                      # todos os eventos
# sondagem = colab.sondar(eventos=[2766, 2977])  # só alguns

## 8. Teste curto de coleta

Baixa só 2 partidas de um evento e gera os Parquet em `vlr_dados/processed/`. Confira o resultado antes da coleta completa.

In [ ]:
colab.coletar([2766], max_partidas=2)
colab.ler('mapas')

In [ ]:
colab.ler('rounds').head()

## 9. Coleta completa

Coloque em `EVENTOS` os IDs do recorte do TCC (veja a tabela da célula 6 ou `docs/sondagem_replay.csv`).

A célula pode rodar por horas. Se o Colab desconectar, rode de novo as células 1, 2 e 4 e depois esta: a coleta continua de onde parou.

In [ ]:
EVENTOS = [
    2766,   # Valorant Champions 2026
    # 2977, # VCT 2026: Americas Stage 2
    # 2976, # VCT 2026: EMEA Stage 2
    # 2776, # VCT 2026: Pacific Stage 2
]
resumo = colab.coletar(EVENTOS)
resumo

## 10. Conferir o resultado

In [ ]:
colab.ler('cobertura_por_evento')

In [ ]:
colab.falhas()        # partidas/eventos que falharam (rode a coleta de novo para refazê-los)

In [ ]:
colab.uso_de_espaco()  # tamanho de cada pasta no Drive

Para regerar as tabelas a partir do cache (por exemplo, depois de trocar a tabela de preços em `config/`), sem baixar nada:

In [ ]:
# colab.processar()

## 11. Encerrar

Garante que todos os arquivos foram enviados ao Drive antes de fechar o notebook.

In [ ]:
colab.finalizar()